# Observability: Metrics, Logs, and Harness Traces

This notebook covers the observability stack for the Custom Deep Research Lab.
You will verify the platform components (Grafana, LokiStack, Prometheus), review
agentic metrics, write LogQL queries, and query the harness `trace_events` table
for per-session metrics.

## 1. Prerequisites

In [ ]:
import os
import subprocess
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

ns = os.getenv("NAMESPACE", "doc-research-lab")
result = subprocess.run(["oc", "get", "namespace", ns], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\u2705 Namespace '{ns}' exists")
else:
    print(f"\u274c Namespace '{ns}' not found \u2014 deploy the application first")

## 2. Verify Observability Stack

Check that LokiStack, the ServiceMonitor, and the GrafanaDashboard CR are deployed.

In [ ]:
checks = [
    ("LokiStack", "oc get lokistack -A -o name"),
    ("ServiceMonitor", f"oc get servicemonitor research-backend -n {ns} -o name"),
    ("GrafanaDashboard", f"oc get grafanadashboard research-lab-metrics -n {ns} -o name"),
]
for name, cmd in checks:
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    status = "\u2705" if result.returncode == 0 else "\u26a0\ufe0f Not found"
    print(f"  {status} {name}")

## 3. Key Agentic Metrics (Prometheus)

The research backend exposes Prometheus metrics on `/metrics`. These are scraped by the
`ServiceMonitor` and visualized in the Grafana dashboard.

| # | Metric | Prometheus Name | Description |
|---|--------|----------------|-------------|
| 1 | Request Rate | `research_requests_total` | Total number of research requests received |
| 2 | Latency Percentiles | `research_request_duration_seconds` | End-to-end request latency (p50, p90, p99) |
| 3 | LLM Token Usage | `llm_tokens_total` | Token consumption by model and type (input/output) |
| 4 | LLM Inference Latency | `llm_inference_duration_seconds` | Time spent in LLM inference calls |
| 5 | Tool Call Success Rate | `tool_calls_total` | MCP tool invocations by name and status (success/error) |
| 6 | Research Quality Scores | `research_quality_score` | LLM-as-Judge quality scores per iteration |

These metrics are sufficient to monitor the health and efficiency of any agentic application.

## 4. Explore the Grafana Dashboard

The `research-lab-metrics` GrafanaDashboard CR deploys a pre-built dashboard.

**How to access:**
1. Open the OpenShift Console
2. Navigate to **Networking \u2192 Routes** in the `doc-research-lab` namespace
3. Click the Grafana route URL
4. Log in with your OpenShift credentials (OAuth proxy)
5. Select the **Research Lab Metrics** dashboard

**Dashboard panels:**

| Row | Panel | Metrics Used |
|-----|-------|---------- ----|
| Overview | Request Rate (req/min) | `rate(research_requests_total[5m])` |
| Overview | Latency Percentiles | `histogram_quantile(0.95, research_request_duration_seconds_bucket)` |
| Overview | Active Requests | `research_active_requests` |
| AI Agent | Token Usage by Model | `sum by(model)(rate(llm_tokens_total[5m]))` |
| AI Agent | LLM Inference Latency | `histogram_quantile(0.95, llm_inference_duration_seconds_bucket)` |
| AI Agent | Tool Call Success Rate | `sum(rate(tool_calls_total{status="success"}[5m]))` |
| AI Agent | Quality Scores Over Time | `research_quality_score` |

## 5. LogQL Queries

LokiStack automatically collects STDOUT and STDERR from all containers running in the
cluster. The research backend uses structured JSON logging (`harness/logging.py`), so
you can parse and filter fields directly in LogQL.

**Where to run these queries:**
- OpenShift Console \u2192 **Observe \u2192 Logs**
- Or the Grafana Explore tab with the Loki data source

In [ ]:
queries = [
    ("All application logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }}'),
    ("Backend logs only",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}", kubernetes_container_name="backend" }} | json | line_format "{{{{.message}}}}"'),
    ("Error logs",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | level="ERROR"'),
    ("Verification results",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | layer="verification"'),
    ("Search operations",
     f'{{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation=~"search.*"'),
    ("Report generation rate",
     f'rate({{ log_type="application", kubernetes_namespace_name="{ns}" }} | json | operation="draft_report" [5m])'),
]

print("LogQL Queries for OpenShift Console \u2192 Observe \u2192 Logs:\n")
for desc, q in queries:
    print(f"\U0001f4cb {desc}:")
    print(f"   {q}\n")
print("\u2705 Copy these queries into the OpenShift Console log viewer")

## 6. Harness Trace Events (PostgreSQL)

The harness records fine-grained trace events to a `trace_events` table in PostgreSQL.
Each event captures session ID, iteration, layer, operation, latency, token usage, and
failure categories. This complements Prometheus metrics with per-session detail.

In [ ]:
import json

try:
    import psycopg2
    _HAS_PSYCOPG2 = True
except ImportError:
    _HAS_PSYCOPG2 = False

if not _HAS_PSYCOPG2:
    print("\u26a0\ufe0f psycopg2 not installed \u2014 pip install psycopg2-binary")
else:
    try:
        conn = psycopg2.connect(
            host=os.getenv("PGVECTOR_HOST", "localhost"),
            port=os.getenv("PGVECTOR_PORT", "5432"),
            dbname=os.getenv("PGVECTOR_DB", "doc_research"),
            user=os.getenv("PGVECTOR_USER", "postgres"),
            password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
        )
        cur = conn.cursor()
        cur.execute("""
            SELECT EXISTS (
                SELECT FROM information_schema.tables
                WHERE table_name = 'trace_events'
            )
        """)
        exists = cur.fetchone()[0]
        if not exists:
            print("\u26a0\ufe0f trace_events table not found \u2014 run a research query first")
            cur.close()
            conn.close()
        else:
            print("\u2705 Connected to PostgreSQL, trace_events table found")
    except Exception as e:
        print(f"\u26a0\ufe0f PostgreSQL unavailable ({e}) \u2014 start with 'make dev-up'")
        _HAS_PSYCOPG2 = False

Query per-session metrics from the `trace_events` table.

In [ ]:
if _HAS_PSYCOPG2 and 'conn' in dir() and conn and not conn.closed:
    cur = conn.cursor()
    cur.execute("""
        SELECT
            session_id,
            COUNT(*) AS total_events,
            SUM(tokens_used) AS total_tokens,
            SUM(latency_ms) AS total_latency_ms,
            COUNT(*) FILTER (WHERE NOT success) AS failures,
            MAX(iteration) AS max_iteration
        FROM trace_events
        GROUP BY session_id
        ORDER BY MAX(timestamp) DESC
        LIMIT 10
    """)
    rows = cur.fetchall()
    if rows:
        print(f"{'Session':>12} {'Events':>7} {'Tokens':>8} {'Latency':>10} {'Fails':>6} {'Iters':>6}")
        print("-" * 55)
        for r in rows:
            print(f"{r[0]:>12} {r[1]:>7} {r[2]:>8} {r[3]:>8}ms {r[4]:>6} {r[5]:>6}")
    else:
        print("No trace events found \u2014 run a research query first")
    cur.close()
    conn.close()
    print("\n\u2705 Harness trace query complete")
else:
    print("\u26a0\ufe0f Skipped \u2014 PostgreSQL not available")

## 7. Summary

In [ ]:
print("\u2705 Observability walkthrough complete")
print()
print("What we covered:")
print("  1. Verified the observability stack (LokiStack, ServiceMonitor, GrafanaDashboard)")
print("  2. Reviewed 6 key agentic Prometheus metrics")
print("  3. Explored the Grafana dashboard panels")
print("  4. Built LogQL queries for application log analysis")
print("  5. Queried harness trace_events for per-session metrics")
print()
print("Next: 3_tracing_mlflow.ipynb \u2014 MLflow tracing for agent spans")